In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# FIXED: Points to your actual validated CRUD module file and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIXED: Configured with your secure credentials for the administrative user account
username = "aacuser"
password = "KeepThisPasswordSafe123!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash
df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# FIXED: Set up the correct file pointer for the Grazioso Salvare brand asset logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Styling header block container to cleanly isolate logo branding elements from data blocks
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()), 
            style={'height': '80px', 'float': 'left', 'padding-right': '20px'}
        ),
        html.H1('Grazioso Salvare Search & Rescue Dashboard', style={'textAlign': 'left', 'margin-top': '10px'}),
        # FIXED: Integrated your unique student identifier directly into layout header
        html.H4('Lead Developer: Athena Trunkhill / SNHU CS-340 Capstone', style={'textAlign': 'left', 'color': '#555'})
    ], style={'padding': '15px', 'background-color': '#F8F9FA', 'overflow': 'auto'}),
    
    html.Hr(),
    
    # FIXED: Added interactive radio options configured to execute targeted rescue filtering profiles
    html.Div([
        html.B("Filter Rescue Animal Profiles: "),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster or Scent Tracking', 'value': 'DISASTER'},
                {'label': 'Reset (Show All Records)', 'value': 'RESET'}
            ],
            value='RESET', # Start state defaults to an unfiltered view
            labelStyle={'display': 'inline-block', 'margin-right': '20px', 'margin-top': '10px'}
        )
    ], style={'padding': '15px'}),
    
    html.Hr(),
    
    # FIXED: Integrated user-friendly parameters into the data table display layer
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        
        # User-friendly native interactions
        sort_action="native",
        filter_action="native",
        page_action="native",
        page_current=0,
        page_size=10,
        
        # Critical structural constraint required for mapping target coordinates
        row_selectable="single",
        selected_rows=[0], 
    ),
    
    html.Br(),
    html.Hr(),
    
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '10px'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%', 'padding': '10px'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# FIXED: Completed dashboard routing callback to query database collection dynamically
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Construct exact search dict lookups using Grazioso Salvare preference tables
    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Newfoundland", "Labrador Retriever Mix", "Chesapeake Bay Retriever"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }
    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd Mix", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky Mix", "Rottweiler"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }
    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "German Shepherd Mix", "Golden Retriever", "Bloodhound", "Beagle Mix"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }
    else:
        query = {} # Reset option falls back to un-queried record arrays
        
    filtered_records = db.read(query)
    filtered_df = pd.DataFrame(filtered_records)
    
    if '_id' in filtered_df.columns:
        filtered_df = filtered_df.drop(columns=['_id'])
        
    return filtered_df.to_dict('records')


# FIXED: Generates an integrated dynamic pie chart summary element reflecting table views
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [dcc.Graph(figure=px.pie(title="No Active Candidate Selection Found"))]
        
    # Re-build frame matching the active controller slice layout view
    current_df = pd.DataFrame.from_dict(viewData)
    
    pie_chart = px.pie(
        current_df, 
        names='breed', 
        title='Breed Distribution for Selected Profile Candidates',
        hole=0.3 # Clean modern donut formatting look
    )
    
    return [dcc.Graph(figure=pie_chart)]

    
# This callback highlights a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback updates the geolocation map view coordinates automatically when a new data row line is checked
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return [dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[dl.TileLayer()])]
    
    dff = pd.DataFrame.from_dict(viewData)
    row = 0 if index is None or len(index) == 0 else index[0]
        
    try:
        # Extract location coordinate values out of row indicators safely using standard column keys
        lat = float(dff.iloc[row]['location_lat'])
        lon = float(dff.iloc[row]['location_long'])
        breed = dff.iloc[row]['breed']
        name = dff.iloc[row]['name']
        
        return [
            dl.Map(style={'width': '100%', 'height': '500px'}, center=[lat, lon], zoom=12, children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(f"Breed: {breed}"),
                    dl.Popup([
                        html.H3("Animal Name"),
                        html.P(name if name and str(name).strip() != "" else "Unnamed Rescue Candidate")
                    ])
                ])
            ])
        ]
    except Exception as e:
        print(f"Mapping engine lookup transformation error: {e}")
        return [dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[dl.TileLayer()])]


# Run app server environment
app.run_server(port=8051)

Successfully connected to MongoDB.
Dash app running on https://helenajoshua-happynato-3000.codio.io/proxy/8051/
